# Lorentz–Floquet: primer experimento

Este cuaderno calcula la monodromía, los multiplicadores de Floquet, el balance de energía y el núcleo causal del prototipo adimensional.

## Modelo material

$$
p''(\tau)+2\zeta p'(\tau)+k(\tau)p(\tau)=f(\tau),
\qquad
k(\tau)=1+m\cos(\nu\tau+\phi).
$$

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Math, display

from causal_em import (
    LorentzParameters,
    causal_susceptibility_grid,
    floquet_exponents,
    integrate_fundamental,
    integrate_trajectory,
    liouville_determinant,
)

## Monodromía y estabilidad de Floquet

$$
\Phi'(\tau)=A(\tau)\Phi(\tau),
\qquad
\Phi(0)=I_2,
\qquad
M=\Phi(T).
$$

Los multiplicadores $\lambda_i\in\sigma(M)$ determinan las tasas reales

$$
g_i=\frac{\log|\lambda_i|}{T}.
$$

In [ ]:
def matrix_latex(matrix):
    rows = [' & '.join(f'{value:.10f}' for value in row) for row in matrix]
    return r'\begin{pmatrix}' + r'\\'.join(rows) + r'\end{pmatrix}'


def complex_latex(value):
    if abs(value.imag) < 1e-12:
        return f'{value.real:.10f}'
    sign = '+' if value.imag >= 0 else '-'
    return f'{value.real:.10f}{sign}{abs(value.imag):.10f}\\mathrm{{i}}'


parameters = LorentzParameters(
    damping_ratio=0.02,
    modulation_depth=0.20,
    modulation_ratio=2.0,
)
result = integrate_fundamental(parameters)
exponents = floquet_exponents(result.matrix, result.period)

display(Math(rf'M={matrix_latex(result.matrix)}'))
display(Math(
    rf'\lambda_1={complex_latex(result.multipliers[0])},\qquad'
    rf'\lambda_2={complex_latex(result.multipliers[1])}'
))
display(Math(
    rf'\mu_1={complex_latex(exponents[0])},\qquad'
    rf'\mu_2={complex_latex(exponents[1])}'
))
display(Math(
    rf'\det M_{{\mathrm{{num}}}}={result.determinant:.14f},\qquad'
    rf'e^{{-2\zeta T}}={liouville_determinant(parameters):.14f}'
))

## Balance de energía

$$
\mathcal{E}(\tau)=\frac12v^2+\frac12k(\tau)p^2,
$$

$$
\frac{d\mathcal{E}}{d\tau}
=-2\zeta v^2+\frac12k'(\tau)p^2+vf(\tau).
$$

In [ ]:
trajectory = integrate_trajectory(parameters, periods=30, samples_per_period=200)
period_index = trajectory.tau / parameters.period

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
axes[0].plot(period_index, trajectory.state[0], label=r'$p$')
axes[0].plot(period_index, trajectory.state[1], label=r"$p'$", alpha=0.8)
axes[0].legend()
axes[0].grid(alpha=0.25)
axes[1].semilogy(period_index, trajectory.energy)
axes[1].set_xlabel(r'Períodos de modulación $n=\tau/T$')
axes[1].set_ylabel(r'Energía $\mathcal{E}$')
axes[1].grid(alpha=0.25)
plt.show()

## Núcleo causal de dos tiempos

$$
g(\tau,s)
=\mathbf e_p^{\mathsf T}U(\tau,s)\mathbf b\,H(\tau-s),
\qquad
g(\tau,s)=0\quad\text{si }\tau\le s.
$$

La periodicidad del medio implica

$$
g(\tau+T,s+T)=g(\tau,s).
$$

In [ ]:
sources = np.linspace(0, 2 * parameters.period, 81)
observations = np.linspace(0, 3 * parameters.period, 121)
kernel = causal_susceptibility_grid(parameters, observations, sources)
limit = np.max(np.abs(kernel))

fig, ax = plt.subplots(figsize=(9, 6))
mesh = ax.pcolormesh(
    sources / parameters.period,
    observations / parameters.period,
    kernel,
    shading='auto', cmap='RdBu_r', vmin=-limit, vmax=limit,
)
diagonal = np.linspace(0, 2, 100)
ax.plot(diagonal, diagonal, 'k--', linewidth=1)
ax.set_xlabel(r'Tiempo fuente $s/T$')
ax.set_ylabel(r'Tiempo de observación $\tau/T$')
ax.set_title(r'Núcleo causal de dos tiempos $g(\tau,s)$')
fig.colorbar(mesh, ax=ax, label='Susceptibilidad adimensional')
plt.show()